In [2]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)

RAW_DIR = '../data/raw'

files = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

dfs = {}
for name, filename in files.items():
    path = os.path.join(RAW_DIR, filename)
    dfs[name] = pd.read_csv(path)
    print(f'{name:22s} loaded  ->  shape: {dfs[name].shape}')

customers              loaded  ->  shape: (99441, 5)
orders                 loaded  ->  shape: (99441, 8)
order_items            loaded  ->  shape: (112650, 7)
payments               loaded  ->  shape: (103886, 5)
reviews                loaded  ->  shape: (99224, 7)
products               loaded  ->  shape: (32951, 9)
sellers                loaded  ->  shape: (3095, 4)
geolocation            loaded  ->  shape: (1000163, 5)
category_translation   loaded  ->  shape: (71, 2)


In [3]:
for name, df in dfs.items():
    print(f'\n===== {name} =====')
    print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
    print(df.dtypes)


===== customers =====
Rows: 99441, Columns: 5
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

===== orders =====
Rows: 99441, Columns: 8
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

===== order_items =====
Rows: 112650, Columns: 7
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

===== payments =====
Rows: 103886, Columns: 5
order_id                    str
payment_sequential        int64
payment_type           

In [4]:
for name, df in dfs.items():
    null_counts = df.isnull().sum()
    null_pct = (df.isnull().mean() * 100).round(2)
    nulls = pd.DataFrame({'missing_count': null_counts, 'missing_pct': null_pct})
    nulls = nulls[nulls['missing_count'] > 0]
    if not nulls.empty:
        print(f'\n===== {name}: columns with missing values =====')
        print(nulls)
    else:
        print(f'\n===== {name}: no missing values =====')


===== customers: no missing values =====

===== orders: columns with missing values =====
                               missing_count  missing_pct
order_approved_at                        160         0.16
order_delivered_carrier_date            1783         1.79
order_delivered_customer_date           2965         2.98

===== order_items: no missing values =====

===== payments: no missing values =====

===== reviews: columns with missing values =====
                        missing_count  missing_pct
review_comment_title            87656        88.34
review_comment_message          58247        58.70

===== products: columns with missing values =====
                            missing_count  missing_pct
product_category_name                 610         1.85
product_name_lenght                   610         1.85
product_description_lenght            610         1.85
product_photos_qty                    610         1.85
product_weight_g                        2         0.01
product_

In [5]:
for name, df in dfs.items():
    dupes = df.duplicated().sum()
    print(f'{name:22s} duplicate rows: {dupes}')

customers              duplicate rows: 0
orders                 duplicate rows: 0
order_items            duplicate rows: 0
payments               duplicate rows: 0
reviews                duplicate rows: 0
products               duplicate rows: 0
sellers                duplicate rows: 0
geolocation            duplicate rows: 261831
category_translation   duplicate rows: 0


In [6]:
cust = dfs['customers']
print('Unique customer_id:', cust['customer_id'].nunique())
print('Unique customer_unique_id:', cust['customer_unique_id'].nunique())
print('Total rows:', len(cust))

Unique customer_id: 99441
Unique customer_unique_id: 96096
Total rows: 99441


In [7]:
print(dfs['orders']['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [8]:
orders = dfs['orders'].copy()
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

impossible = orders[orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']]
print('Orders delivered before purchase date:', len(impossible))

Orders delivered before purchase date: 0


In [9]:
items = dfs['order_items']
print('Rows with price <= 0:', (items['price'] <= 0).sum())
print('Rows with freight_value < 0:', (items['freight_value'] < 0).sum())
print(items[['price', 'freight_value']].describe())

Rows with price <= 0: 0
Rows with freight_value < 0: 0
               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000


In [10]:
orphan_products = ~items['product_id'].isin(dfs['products']['product_id'])
orphan_sellers = ~items['seller_id'].isin(dfs['sellers']['seller_id'])
print('order_items rows with product_id not in products table:', orphan_products.sum())
print('order_items rows with seller_id not in sellers table:', orphan_sellers.sum())

order_items rows with product_id not in products table: 0
order_items rows with seller_id not in sellers table: 0


In [11]:
for name, df in dfs.items():
    print(f'\n===== {name} sample =====')
    display(df.head(3))


===== customers sample =====


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



===== orders sample =====


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



===== order_items sample =====


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



===== geolocation sample =====


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP



===== category_translation sample =====


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [12]:
for col in ['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']:
    print(f'\n=== Missing {col} by order_status ===')
    print(orders[orders[col].isnull()]['order_status'].value_counts())


=== Missing order_approved_at by order_status ===
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

=== Missing order_delivered_carrier_date by order_status ===
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

=== Missing order_delivered_customer_date by order_status ===
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [13]:
products = dfs['products']
cols_to_check = ['product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty']
missing_sets = {c: set(products[products[c].isnull()].index) for c in cols_to_check}
print('All four columns missing on the exact same rows:', missing_sets['product_category_name'] == missing_sets['product_name_lenght'] == missing_sets['product_description_lenght'] == missing_sets['product_photos_qty'])

All four columns missing on the exact same rows: True


In [14]:
items = dfs['order_items']
zero_freight = items[items['freight_value'] == 0]
print('Rows with freight_value == 0:', len(zero_freight))
print('Distinct sellers among them:', zero_freight['seller_id'].nunique())
print('Distinct products among them:', zero_freight['product_id'].nunique())

Rows with freight_value == 0: 383
Distinct sellers among them: 9
Distinct products among them: 10


In [15]:
delivered = orders[orders['order_status'] == 'delivered']

print("Delivered orders missing order_approved_at:")
display(delivered[delivered['order_approved_at'].isnull()])

print("\nDelivered orders missing order_delivered_carrier_date:")
display(delivered[delivered['order_delivered_carrier_date'].isnull()])

print("\nDelivered orders missing order_delivered_customer_date:")
display(delivered[delivered['order_delivered_customer_date'].isnull()])


Delivered orders missing order_approved_at:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaT,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:47,NaT,2017-02-23 03:11:48,2017-03-02 03:41:58,2017-03-27
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
48401,7002a78c79c519ac54022d4f8a65e6e8,d5de688c321096d15508faae67a27051,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
61743,2eecb0d85f281280f79fa00f9cec1a95,a3d3c38e58b9d2dfb9207cab690b6310,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20



Delivered orders missing order_delivered_carrier_date:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23



Delivered orders missing order_delivered_customer_date:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [16]:
print("Purchase dates - missing carrier_date:")
print(delivered[delivered['order_delivered_carrier_date'].isnull()]['order_purchase_timestamp'].dt.date)

print("\nPurchase dates - missing customer_date:")
print(delivered[delivered['order_delivered_customer_date'].isnull()]['order_purchase_timestamp'].dt.date)

Purchase dates - missing carrier_date:
73222    2017-09-29
92643    2017-05-25
Name: order_purchase_timestamp, dtype: object

Purchase dates - missing customer_date:
3002     2017-11-28
20618    2018-06-20
43834    2018-07-01
79263    2018-07-01
82868    2018-07-01
92643    2017-05-25
97647    2018-06-08
98038    2018-06-27
Name: order_purchase_timestamp, dtype: object
